# Notebook 08 — Front Tracker Benchmark

**Objective:** compare the accepted ByteTrack B15 baseline with BoT-SORT under the same detector, video, image size, device, detection floor, high-confidence threshold, and visibility ground truth.

Scientific limitation: the benchmark video contains one fish. It supports diagnostics of continuity, temporal gaps, duplicate tracks, shelter-related interruption, and workstation speed, but does not adequately test multi-fish identity association, crossing-induced ID changes, or occlusion between fish. It cannot establish the best tracker for multi-fish MOT. Reported transitions are not official MOT ID switches.

## 1. Experiment metadata and CONFIG

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.util, json, os, platform, subprocess, sys, time
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

STARTED_AT = datetime.now(timezone.utc).isoformat()
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CUDA_AVAILABLE = torch.cuda.is_available(); GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'
EXPERIMENT_ID = 'FRONT_TRACKER_BENCHMARK_001'
VIDEO_PATH = PROJECT_ROOT / 'data' / 'raw' / 'front' / '4.mp4'
MODEL_PATH = PROJECT_ROOT / 'runs' / 'front' / 'yolov8n_front_v1_baseline' / 'weights' / 'best.pt'
EXPECTED_MODEL_SHA256 = '750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738'
VISIBILITY_PATH = PROJECT_ROOT / 'results' / 'detection' / 'front_visibility_frame_labels.csv'
SOURCE_DETECTION_ID = 'FRONT_VIDEO_DET_CONF068_N1_001'
DETECTOR_FLOOR = 0.50
HIGH_CONFIDENCE_THRESHOLD = 0.68
NMS_IOU = 0.70
IMGSZ = 640
DEVICE = 0
EXPECTED_FISH_COUNT = 1
PROGRESS_INTERVAL = 200
FLOAT_TOLERANCE = 1e-6
TRACKERS = {'BYTE_B15': PROJECT_ROOT / 'configs' / 'trackers' / 'front_bytetrack_b15.yaml', 'BOTSORT': PROJECT_ROOT / 'configs' / 'trackers' / 'front_botsort_benchmark.yaml'}
TRACKER_COMPLEXITY_RANK = {'BYTE_B15': 0, 'BOTSORT': 1}
OTHER_OFFICIAL_CONFIGS_NOT_BENCHMARKED = ['deepocsort.yaml', 'fasttrack.yaml', 'ocsort.yaml', 'tracktrack.yaml']
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'front' / 'tracking' / 'benchmark'
RESULT_PATH = PROJECT_ROOT / 'results' / 'tracking' / 'front_tracker_benchmark.csv'
LOG_DIR = PROJECT_ROOT / 'logs' / 'tracking' / EXPERIMENT_ID
SOURCE_CONFIG_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_DETECTION_ID / 'config.yaml'
SOURCE_SUMMARY_PATH = PROJECT_ROOT / 'logs' / 'detection' / SOURCE_DETECTION_ID / 'summary.json'
CONFIG = {'experiment_id': EXPERIMENT_ID, 'video': str(VIDEO_PATH.relative_to(PROJECT_ROOT)), 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'visibility_groundtruth': str(VISIBILITY_PATH.relative_to(PROJECT_ROOT)), 'detector_floor': DETECTOR_FLOOR, 'high_confidence_threshold': HIGH_CONFIDENCE_THRESHOLD, 'nms_iou': NMS_IOU, 'imgsz': IMGSZ, 'device': DEVICE, 'expected_fish_count': EXPECTED_FISH_COUNT, 'trackers': {name: str(path.relative_to(PROJECT_ROOT)) for name, path in TRACKERS.items()}, 'output_root': str(OUTPUT_ROOT.relative_to(PROJECT_ROOT)), 'result_path': str(RESULT_PATH.relative_to(PROJECT_ROOT))}
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}; Conda environment: {CONDA_ENV}')
print(f'Python: {platform.python_version()}; Torch: {torch.__version__}; Ultralytics: {ultralytics.__version__}')
print(f'Device: {DEVICE}; CUDA available: {CUDA_AVAILABLE}; GPU: {GPU_NAME}; Git commit: {GIT_COMMIT}')
print('CONFIG — FRONT TRACKER BENCHMARK')
for key, value in CONFIG.items(): print(f'{key}: {value}')
print(f'Official Ultralytics configs available but NOT benchmarked without USER approval: {OTHER_OFFICIAL_CONFIGS_NOT_BENCHMARKED}')
if importlib.util.find_spec('lap') is None: raise ModuleNotFoundError('MISSING_DEPENDENCY: tracker benchmark requires lap>=0.5.12 in Conda env fish.')

experiment_id: FRONT_TRACKER_BENCHMARK_001
datetime_utc: 2026-08-17T12:55:47.603851+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python; Conda environment: fish
Python: 3.11.15; Torch: 2.13.0+cu130; Ultralytics: 8.4.120
Device: 0; CUDA available: True; GPU: NVIDIA GeForce RTX 3050; Git commit: 78ed4c11909367c4eef4724991449f74994d917f
CONFIG — FRONT TRACKER BENCHMARK
experiment_id: FRONT_TRACKER_BENCHMARK_001
video: data/raw/front/4.mp4
model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
visibility_groundtruth: results/detection/front_visibility_frame_labels.csv
detector_floor: 0.5
high_confidence_threshold: 0.68
nms_iou: 0.7
imgsz: 640
device: 0
expected_fish_count: 1
trackers: {'BYTE_B15': 'configs/trackers/front_bytetrack_b15.yaml', 'BOTSORT': 'configs/trackers/front_botsort_benchmark.yaml'}
output_root: outputs/front/tracking/benchmark
result_path: results/tracking/front_tracker_benchmark.csv
Official Ultralytics configs av

## 2. Fair-comparison provenance and tracker-config preflight

In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()
assert CONDA_ENV == 'fish' and CUDA_AVAILABLE
assert ultralytics.__version__ == '8.4.120', f'FAIL provenance: expected Ultralytics 8.4.120, got {ultralytics.__version__}'
assert 0.0 <= DETECTOR_FLOOR <= HIGH_CONFIDENCE_THRESHOLD <= 1.0
for path in (VIDEO_PATH, MODEL_PATH, VISIBILITY_PATH, SOURCE_CONFIG_PATH, SOURCE_SUMMARY_PATH, *TRACKERS.values()): assert path.is_file(), f'FAIL preflight: missing {path.relative_to(PROJECT_ROOT)}'
MODEL_SHA256 = sha256_file(MODEL_PATH); assert MODEL_SHA256 == EXPECTED_MODEL_SHA256
SOURCE_CONFIG = yaml.safe_load(SOURCE_CONFIG_PATH.read_text(encoding='utf-8')); SOURCE_SUMMARY = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
VIDEO_SHA256 = sha256_file(VIDEO_PATH)
assert VIDEO_PATH == PROJECT_ROOT / SOURCE_CONFIG['video_path']
assert VIDEO_SHA256 == SOURCE_CONFIG['video_sha256'] == SOURCE_SUMMARY['video_sha256']
assert MODEL_SHA256 == SOURCE_SUMMARY['model_sha256']
assert np.isclose(float(SOURCE_CONFIG['nms_iou']), NMS_IOU) and int(SOURCE_CONFIG['imgsz']) == IMGSZ
VIDEO_FPS = float(SOURCE_CONFIG['video_fps']); TOTAL_FRAMES = int(SOURCE_CONFIG['video_frame_count']); VIDEO_WIDTH, VIDEO_HEIGHT = map(int, SOURCE_CONFIG['video_resolution'].split('x'))
VISIBILITY = pd.read_csv(VISIBILITY_PATH).sort_values('frame_index').reset_index(drop=True)
assert {'frame_index', 'time_sec', 'visibility_state'}.issubset(VISIBILITY.columns)
assert len(VISIBILITY) == TOTAL_FRAMES and np.array_equal(VISIBILITY['frame_index'].to_numpy(), np.arange(TOTAL_FRAMES))
VISIBILITY['visible_interval_id'] = pd.Series(pd.NA, index=VISIBILITY.index, dtype='Int64')
interval_id = 0; previous_visible = False
for row_index, state in enumerate(VISIBILITY['visibility_state']):
    current_visible = state == 'VISIBLE'
    if current_visible and not previous_visible: interval_id += 1
    if current_visible: VISIBILITY.at[row_index, 'visible_interval_id'] = interval_id
    previous_visible = current_visible
VISIBLE_FRAMES = int((VISIBILITY['visibility_state'] == 'VISIBLE').sum()); assert VISIBLE_FRAMES > 0
REQUESTED_CONFIGS = {}
for label, path in TRACKERS.items():
    config = yaml.safe_load(path.read_text(encoding='utf-8'))
    assert config['tracker_type'] == ('bytetrack' if label == 'BYTE_B15' else 'botsort')
    assert np.isclose(float(config['track_low_thresh']), DETECTOR_FLOOR)
    assert np.isclose(float(config['track_high_thresh']), HIGH_CONFIDENCE_THRESHOLD)
    assert np.isclose(float(config['new_track_thresh']), HIGH_CONFIDENCE_THRESHOLD)
    assert 0.0 <= float(config['track_low_thresh']) <= float(config['track_high_thresh']) <= 1.0
    if label == 'BYTE_B15': assert int(config['track_buffer']) == 15
    if label == 'BOTSORT':
        assert int(config['track_buffer']) == 30 and config['gmc_method'] == 'sparseOptFlow' and bool(config['with_reid']) is False and config['model'] == 'auto'
    REQUESTED_CONFIGS[label] = config
for label in TRACKERS:
    output_dir = OUTPUT_ROOT / label
    if output_dir.exists() and any(output_dir.iterdir()): raise RuntimeError(f'FAIL preflight: preserve existing output {output_dir.relative_to(PROJECT_ROOT)}')
if RESULT_PATH.exists() or (LOG_DIR.exists() and any(LOG_DIR.iterdir())): raise RuntimeError('FAIL preflight: preserve existing benchmark evidence before rerun.')
print(f'Video SHA-256: {VIDEO_SHA256}; model SHA-256: {MODEL_SHA256}')
print(f'Frames={TOTAL_FRAMES}; FPS={VIDEO_FPS:.6f}; visible frames={VISIBLE_FRAMES}; visible intervals={interval_id}')
for label, config in REQUESTED_CONFIGS.items(): print(f'{label} REQUESTED CONFIG: {config}')
print('FAIR-COMPARISON PREFLIGHT: PASS')

Video SHA-256: 3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700; model SHA-256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
Frames=3431; FPS=28.668432; visible frames=2335; visible intervals=18
BYTE_B15 REQUESTED CONFIG: {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
BOTSORT REQUESTED CONFIG: {'tracker_type': 'botsort', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 30, 'match_thresh': 0.8, 'fuse_score': True, 'gmc_method': 'sparseOptFlow', 'proximity_thresh': 0.5, 'appearance_thresh': 0.8, 'with_reid': False, 'model': 'auto'}
FAIR-COMPARISON PREFLIGHT: PASS


## 3. Run BYTE_B15 and BOTSORT on identical inputs

In [3]:
from ultralytics import YOLO
TRACKING_TABLES = {}; PROCESSING_STATS = {}; ACTUAL_CONFIGS = {}
for label, tracker_path in TRACKERS.items():
    output_dir = OUTPUT_ROOT / label; output_dir.mkdir(parents=True, exist_ok=True)
    tracks_path = output_dir / 'frame_tracks.csv'; overlay_path = output_dir / 'overlay.mp4'
    model = YOLO(str(MODEL_PATH), task='detect')
    capture = cv2.VideoCapture(str(VIDEO_PATH)); assert capture.isOpened(), f'FAIL {label}: video open'
    writer = cv2.VideoWriter(str(overlay_path), cv2.VideoWriter_fourcc(*'mp4v'), VIDEO_FPS, (VIDEO_WIDTH, VIDEO_HEIGHT))
    if not writer.isOpened(): capture.release(); raise RuntimeError(f'FAIL {label}: overlay writer')
    rows = []; frame_index = 0; checked = False; run_start = time.perf_counter()
    print(f'{label} START — detector floor={DETECTOR_FLOOR}; high/new threshold={HIGH_CONFIDENCE_THRESHOLD}; imgsz={IMGSZ}; device={DEVICE}')
    try:
        while True:
            ok, frame = capture.read()
            if not ok: break
            time_sec = frame_index / VIDEO_FPS; state = VISIBILITY.iloc[frame_index]['visibility_state']
            result = model.track(source=frame, persist=True, tracker=str(tracker_path), conf=DETECTOR_FLOOR, iou=NMS_IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
            if not checked:
                actual_tracker = model.predictor.trackers[0]
                actual = {key: getattr(actual_tracker.args, key) for key in REQUESTED_CONFIGS[label]}
                for key, requested in REQUESTED_CONFIGS[label].items():
                    actual_value = actual[key]
                    if isinstance(requested, float): assert np.isclose(float(actual_value), requested), f'FAIL {label}: actual {key}'
                    else: assert actual_value == requested, f'FAIL {label}: actual {key}={actual_value!r}, requested={requested!r}'
                assert float(actual['track_low_thresh']) >= DETECTOR_FLOOR
                ACTUAL_CONFIGS[label] = actual; print(f'{label} ACTUAL CONFIG: {actual}'); checked = True
            boxes = result.boxes; observed = boxes is not None and boxes.id is not None and len(boxes.id) > 0; overlay = frame.copy()
            if observed:
                ids = boxes.id.detach().cpu().numpy().astype(int); confs = boxes.conf.detach().cpu().numpy().astype(float); coords = boxes.xyxy.detach().cpu().numpy().astype(float)
                assert float(confs.min()) >= DETECTOR_FLOOR - FLOAT_TOLERANCE
                for track_id, confidence, xyxy in zip(ids, confs, coords):
                    x1, y1, x2, y2 = xyxy.tolist(); cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                    rows.append({'frame_index': frame_index, 'time_sec': time_sec, 'visibility_state': state, 'track_id': int(track_id), 'confidence': float(confidence), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'cx': cx, 'cy': cy})
                    p1, p2 = (int(round(x1)), int(round(y1))), (int(round(x2)), int(round(y2)))
                    cv2.rectangle(overlay, p1, p2, (0, 220, 0), 2); cv2.putText(overlay, f'ID {track_id} | {confidence:.2f}', (p1[0], max(18, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2, cv2.LINE_AA)
            else:
                rows.append({'frame_index': frame_index, 'time_sec': time_sec, 'visibility_state': state, 'track_id': np.nan, 'confidence': np.nan, 'x1': np.nan, 'y1': np.nan, 'x2': np.nan, 'y2': np.nan, 'cx': np.nan, 'cy': np.nan})
            active = int(boxes.id.numel()) if observed else 0
            header = f'{label} | frame={frame_index}/{TOTAL_FRAMES - 1} | t={time_sec:.2f}s | state={state} | active={active}'
            cv2.rectangle(overlay, (0, 0), (min(VIDEO_WIDTH, 1000), 36), (0, 0, 0), -1); cv2.putText(overlay, header, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.64, (255, 255, 255), 2, cv2.LINE_AA)
            writer.write(overlay); frame_index += 1
            if frame_index % PROGRESS_INTERVAL == 0 or frame_index == TOTAL_FRAMES:
                elapsed = time.perf_counter() - run_start; print(f'{label}: {frame_index}/{TOTAL_FRAMES} | elapsed={elapsed:.1f}s | FPS={frame_index / elapsed:.2f}')
    finally:
        capture.release(); writer.release()
    runtime = time.perf_counter() - run_start
    assert frame_index == TOTAL_FRAMES and checked
    table = pd.DataFrame(rows, columns=['frame_index', 'time_sec', 'visibility_state', 'track_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy'])
    assert table['frame_index'].nunique() == TOTAL_FRAMES
    table.to_csv(tracks_path, index=False); TRACKING_TABLES[label] = table
    PROCESSING_STATS[label] = {'runtime_sec': runtime, 'processing_FPS': TOTAL_FRAMES / runtime, 'tracks_path': str(tracks_path.relative_to(PROJECT_ROOT)), 'overlay_path': str(overlay_path.relative_to(PROJECT_ROOT))}
    print(f'{label} COMPLETE — rows={len(table)}; workstation FPS={TOTAL_FRAMES / runtime:.2f}')

BYTE_B15 START — detector floor=0.5; high/new threshold=0.68; imgsz=640; device=0
BYTE_B15 ACTUAL CONFIG: {'tracker_type': 'bytetrack', 'track_high_thresh': 0.68, 'track_low_thresh': 0.5, 'new_track_thresh': 0.68, 'track_buffer': 15, 'match_thresh': 0.8, 'fuse_score': True}
BYTE_B15: 200/3431 | elapsed=6.6s | FPS=30.32
BYTE_B15: 400/3431 | elapsed=10.9s | FPS=36.85
BYTE_B15: 600/3431 | elapsed=15.4s | FPS=38.98
BYTE_B15: 800/3431 | elapsed=19.7s | FPS=40.70
BYTE_B15: 1000/3431 | elapsed=23.8s | FPS=42.02
BYTE_B15: 1200/3431 | elapsed=27.3s | FPS=43.91
BYTE_B15: 1400/3431 | elapsed=30.9s | FPS=45.33
BYTE_B15: 1600/3431 | elapsed=34.3s | FPS=46.61
BYTE_B15: 1800/3431 | elapsed=37.9s | FPS=47.45
BYTE_B15: 2000/3431 | elapsed=41.5s | FPS=48.23
BYTE_B15: 2200/3431 | elapsed=45.4s | FPS=48.42
BYTE_B15: 2400/3431 | elapsed=49.5s | FPS=48.52
BYTE_B15: 2600/3431 | elapsed=53.6s | FPS=48.55
BYTE_B15: 2800/3431 | elapsed=57.6s | FPS=48.64
BYTE_B15: 3000/3431 | elapsed=61.6s | FPS=48.72
BYTE_B15: 

## 4. Visibility-aware continuity, lifespan, and shelter diagnostics

The first track segment in each independent VISIBLE interval is expected. `visible_excess_fragments` counts only additional segments within that interval. ID transitions never cross `IN_SHELTER`, `PARTIALLY_VISIBLE`, or `UNCERTAIN`. Duplicate tracks are diagnostics, not official false positives.

In [4]:
def consecutive_run_lengths(indices):
    values = np.asarray(indices, dtype=int)
    if not len(values): return []
    lengths = []; start = previous = int(values[0])
    for value in values[1:]:
        value = int(value)
        if value != previous + 1: lengths.append(previous - start + 1); start = value
        previous = value
    lengths.append(previous - start + 1); return lengths
COMPARISON_ROWS = []; INTERVAL_TABLES = {}; LIFESPAN_TABLES = {}
for label, table in TRACKING_TABLES.items():
    observed = table[table['track_id'].notna()].copy(); observed['track_id'] = observed['track_id'].astype(int)
    counts = observed.groupby('frame_index').size().reindex(np.arange(TOTAL_FRAMES), fill_value=0).astype(int)
    status = VISIBILITY[['frame_index', 'time_sec', 'visibility_state', 'visible_interval_id']].copy(); status['active_track_count'] = counts.to_numpy()
    visible = status[status['visibility_state'] == 'VISIBLE'].copy()
    tracked_frames = int((visible['active_track_count'] >= 1).sum()); untracked_frames = int((visible['active_track_count'] == 0).sum())
    one_frames = int((visible['active_track_count'] == EXPECTED_FISH_COUNT).sum()); multi_frames = int((visible['active_track_count'] > EXPECTED_FISH_COUNT).sum())
    total_segments = 0; excess_fragments = 0; transitions_total = 0; untracked_lengths = []; interval_rows = []
    for visible_id, interval in visible.groupby('visible_interval_id', sort=True):
        frame_ids = interval['frame_index'].to_numpy(dtype=int); interval_obs = observed[observed['frame_index'].isin(frame_ids)]
        segments = 0; transitions = 0; active_segment_id = None; last_observed_id = None
        for frame_index in frame_ids:
            ids = interval_obs.loc[interval_obs['frame_index'] == frame_index, 'track_id'].tolist()
            if len(ids) == EXPECTED_FISH_COUNT:
                current_id = int(ids[0])
                if active_segment_id != current_id: segments += 1; active_segment_id = current_id
                if last_observed_id is not None and current_id != last_observed_id: transitions += 1
                last_observed_id = current_id
            else: active_segment_id = None
        local_untracked = consecutive_run_lengths(interval.loc[interval['active_track_count'] == 0, 'frame_index'])
        local_excess = max(0, segments - 1)
        total_segments += segments; excess_fragments += local_excess; transitions_total += transitions; untracked_lengths.extend(local_untracked)
        interval_rows.append({'visible_interval_id': int(visible_id), 'start_frame': int(frame_ids.min()), 'end_frame': int(frame_ids.max()), 'visible_frames': len(frame_ids), 'unique_track_ids_visible': int(interval_obs['track_id'].nunique()), 'visible_id_transitions': transitions, 'track_segments_in_interval': segments, 'excess_track_fragments': local_excess, 'longest_untracked_run_frames': max(local_untracked, default=0)})
    interval_df = pd.DataFrame(interval_rows); INTERVAL_TABLES[label] = interval_df
    lifespans = observed.groupby('track_id').agg(first_frame=('frame_index', 'min'), last_frame=('frame_index', 'max'), observed_frames=('frame_index', 'nunique')).reset_index(); lifespans['lifespan_sec'] = lifespans['observed_frames'] / VIDEO_FPS; LIFESPAN_TABLES[label] = lifespans
    longest_gap = max(untracked_lengths, default=0)
    COMPARISON_ROWS.append({'tracker': label, 'visible_frames': VISIBLE_FRAMES, 'visible_tracked_frames': tracked_frames, 'visible_untracked_frames': untracked_frames, 'visible_track_coverage': tracked_frames / VISIBLE_FRAMES, 'visible_zero_track_rate': untracked_frames / VISIBLE_FRAMES, 'visible_one_track_rate': one_frames / VISIBLE_FRAMES, 'visible_multi_track_rate': multi_frames / VISIBLE_FRAMES, 'visible_interval_count': int(visible['visible_interval_id'].nunique()), 'visible_track_segments_total': total_segments, 'visible_excess_fragments': excess_fragments, 'visible_id_transitions': transitions_total, 'longest_visible_untracked_run_frames': longest_gap, 'longest_visible_untracked_run_sec': longest_gap / VIDEO_FPS, 'median_visible_untracked_run_frames': float(np.median(untracked_lengths)) if untracked_lengths else 0.0, 'p95_visible_untracked_run_frames': float(np.quantile(untracked_lengths, 0.95)) if untracked_lengths else 0.0, 'tracks_observed_during_shelter': int((observed['visibility_state'] == 'IN_SHELTER').sum()), 'unique_track_ids_total': int(observed['track_id'].nunique()), 'median_track_lifespan_sec': float(lifespans['lifespan_sec'].median()) if len(lifespans) else 0.0, 'mean_track_lifespan_sec': float(lifespans['lifespan_sec'].mean()) if len(lifespans) else 0.0, 'max_track_lifespan_sec': float(lifespans['lifespan_sec'].max()) if len(lifespans) else 0.0, 'processing_FPS': PROCESSING_STATS[label]['processing_FPS']})
COMPARISON_DF = pd.DataFrame(COMPARISON_ROWS)
display(COMPARISON_DF)
print('processing_FPS is measured on this workstation, not Raspberry Pi hardware.')

,tracker,visible_frames,visible_tracked_frames,visible_untracked_frames,visible_track_coverage,visible_zero_track_rate,visible_one_track_rate,visible_multi_track_rate,visible_interval_count,visible_track_segments_total,...,longest_visible_untracked_run_frames,longest_visible_untracked_run_sec,median_visible_untracked_run_frames,p95_visible_untracked_run_frames,tracks_observed_during_shelter,unique_track_ids_total,median_track_lifespan_sec,mean_track_lifespan_sec,max_track_lifespan_sec,processing_FPS
0,BYTE_B15,2335,2333,2,0.999143,0.000857,0.999143,0.0,18,19,...,1,0.034882,1.0,1.0,6,2,41.509072,41.509072,49.148136,48.776541
1,BOTSORT,2335,2333,2,0.999143,0.000857,0.999143,0.0,18,19,...,1,0.034882,1.0,1.0,6,2,41.509072,41.509072,49.148136,31.531344


processing_FPS is measured on this workstation, not Raspberry Pi hardware.


## 5. Selection candidate and evidence

In [5]:
ranked = COMPARISON_DF.assign(complexity_rank=COMPARISON_DF['tracker'].map(TRACKER_COMPLEXITY_RANK)).sort_values(['visible_track_coverage', 'visible_excess_fragments', 'visible_id_transitions', 'visible_multi_track_rate', 'longest_visible_untracked_run_frames', 'complexity_rank', 'processing_FPS'], ascending=[False, True, True, True, True, True, False]).reset_index(drop=True)
BEST_CANDIDATE = str(ranked.iloc[0]['tracker'])
SELECTION_REASON = 'Provisional ranking prioritizes visibility-aware quality; if quality is equivalent, the simpler/lighter tracker is preferred for Raspberry Pi deployment. USER decides the final tracker.'
WARNINGS = ['This one-fish benchmark does not establish multi-fish MOT performance.']
for row in COMPARISON_DF.itertuples(index=False):
    if row.visible_track_coverage < 1.0 or row.visible_excess_fragments > 0 or row.visible_id_transitions > 0 or row.visible_multi_track_rate > 0 or row.tracks_observed_during_shelter > 0: WARNINGS.append(f'{row.tracker} has imperfect diagnostic values; inspect its metrics and overlay.')
CHECKPOINT_RESULT = 'PASS_WITH_WARNING' if WARNINGS else 'PASS'
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True); LOG_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DF.to_csv(RESULT_PATH, index=False)
for label in TRACKERS:
    INTERVAL_TABLES[label].to_csv(OUTPUT_ROOT / label / 'visible_interval_diagnostics.csv', index=False)
    LIFESPAN_TABLES[label].to_csv(OUTPUT_ROOT / label / 'track_lifespans.csv', index=False)
CONFIG_PATH = LOG_DIR / 'config.yaml'; ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'; SUMMARY_PATH = LOG_DIR / 'summary.json'
CONFIG_EVIDENCE = {**CONFIG, 'model_sha256': MODEL_SHA256, 'video_sha256': VIDEO_SHA256, 'video_fps': VIDEO_FPS, 'total_frames': TOTAL_FRAMES, 'visibility_groundtruth_sha256': sha256_file(VISIBILITY_PATH), 'ultralytics_version': ultralytics.__version__, 'requested_tracker_configs': REQUESTED_CONFIGS, 'actual_tracker_configs': ACTUAL_CONFIGS, 'other_official_configs_not_benchmarked': OTHER_OFFICIAL_CONFIGS_NOT_BENCHMARKED, 'git_commit': GIT_COMMIT}
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}', f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}', f'torch={torch.__version__}', f'cuda_runtime={torch.version.cuda}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}', f'opencv={cv2.__version__}']
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
OUTPUT_FILES = [RESULT_PATH, CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH] + [OUTPUT_ROOT / label / filename for label in TRACKERS for filename in ('frame_tracks.csv', 'overlay.mp4', 'visible_interval_diagnostics.csv', 'track_lifespans.csv')]
SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model_sha256': MODEL_SHA256, 'video_sha256': VIDEO_SHA256, 'visibility_groundtruth': 'loaded', 'trackers': {row.tracker: row._asdict() for row in COMPARISON_DF.itertuples(index=False)}, 'requested_tracker_configs': REQUESTED_CONFIGS, 'actual_tracker_configs': ACTUAL_CONFIGS, 'best_candidate': BEST_CANDIDATE, 'selection_reason': SELECTION_REASON, 'scientific_limitation': 'One-fish video does not fully evaluate multi-fish identity association, crossing-induced ID switches, or inter-fish occlusion.', 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'git_commit': GIT_COMMIT, 'next_step': 'USER reviews tracker benchmark before any Notebook 09 work.'}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
for path in OUTPUT_FILES: print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

Created results/tracking/front_tracker_benchmark.csv (950 bytes)
Created logs/tracking/FRONT_TRACKER_BENCHMARK_001/config.yaml (2057 bytes)
Created logs/tracking/FRONT_TRACKER_BENCHMARK_001/environment.txt (353 bytes)
Created logs/tracking/FRONT_TRACKER_BENCHMARK_001/summary.json (5229 bytes)
Created outputs/front/tracking/benchmark/BYTE_B15/frame_tracks.csv (427496 bytes)
Created outputs/front/tracking/benchmark/BYTE_B15/overlay.mp4 (75171059 bytes)
Created outputs/front/tracking/benchmark/BYTE_B15/visible_interval_diagnostics.csv (627 bytes)
Created outputs/front/tracking/benchmark/BYTE_B15/track_lifespans.csv (126 bytes)
Created outputs/front/tracking/benchmark/BOTSORT/frame_tracks.csv (427548 bytes)
Created outputs/front/tracking/benchmark/BOTSORT/overlay.mp4 (75108576 bytes)
Created outputs/front/tracking/benchmark/BOTSORT/visible_interval_diagnostics.csv (627 bytes)
Created outputs/front/tracking/benchmark/BOTSORT/track_lifespans.csv (126 bytes)


## 6. Final Summary

In [6]:
FINAL_SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model_sha256': MODEL_SHA256, 'video_sha256': VIDEO_SHA256, 'visibility_groundtruth': 'loaded'}
for label in TRACKERS:
    row = COMPARISON_DF.set_index('tracker').loc[label]
    FINAL_SUMMARY[label] = {'visible_track_coverage': row.visible_track_coverage, 'visible_excess_fragments': int(row.visible_excess_fragments), 'visible_id_transitions': int(row.visible_id_transitions), 'visible_multi_track_rate': row.visible_multi_track_rate, 'longest_visible_untracked_run_sec': row.longest_visible_untracked_run_sec, 'unique_track_ids_total': int(row.unique_track_ids_total), 'processing_FPS': row.processing_FPS}
FINAL_SUMMARY.update({'best_candidate': BEST_CANDIDATE, 'selection_reason': SELECTION_REASON, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'next_step': 'USER reviews tracker benchmark before any Notebook 09 work.'})
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items(): print(f'{key}: {value}')

FINAL SUMMARY
experiment_id: FRONT_TRACKER_BENCHMARK_001
model_sha256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
video_sha256: 3f8587344beba8dcc6f835d1ed94aa8daadb06a369b5b96977ee902f035b5700
visibility_groundtruth: loaded
BYTE_B15: {'visible_track_coverage': np.float64(0.9991434689507495), 'visible_excess_fragments': 1, 'visible_id_transitions': 0, 'visible_multi_track_rate': np.float64(0.0), 'longest_visible_untracked_run_sec': np.float64(0.03488157291363062), 'unique_track_ids_total': 2, 'processing_FPS': np.float64(48.77654058298213)}
BOTSORT: {'visible_track_coverage': np.float64(0.9991434689507495), 'visible_excess_fragments': 1, 'visible_id_transitions': 0, 'visible_multi_track_rate': np.float64(0.0), 'longest_visible_untracked_run_sec': np.float64(0.03488157291363062), 'unique_track_ids_total': 2, 'processing_FPS': np.float64(31.531343907695245)}
best_candidate: BYTE_B15
selection_reason: Provisional ranking prioritizes visibility-aware quality; if qualit